In [ ]:
#Columns used
#Furnace Temp, Carbon Content, Oxygen Levels, Silicon Content, Manganese Content, Sulfur Content, Phosphorus Content, Scrap Ratio, Power input, Tap Weight
#Range
# 1550-1650, 0.02-0.8, 0.5-3.0, 0.1-0.4, 0.3-1.5, 0-0.05, 0-0.04,20-80, 300-600, 100-300 
#Output steel grade

import numpy as np
import pandas as pd

n_hours = 24

data = {
    'Furnace_Temp': np.random.uniform(1550, 1650, n_hours),
    'Carbon_Content': np.random.uniform(0.02, 0.8, n_hours),
    'Oxygen Levels': np.random.uniform(0.5, 3.0, n_hours),
    'Silicon Content': np.random.uniform(0.1, 0.4, n_hours),
    'Manganese Content': np.random.uniform(0.3, 1.5, n_hours),
    'Sulfur Content': np.random.uniform(0.0, 0.05, n_hours),
    'Phosphorus Content': np.random.uniform(0.0, 0.04, n_hours),
    'Scrap Ratio': np.random.uniform(20, 80, n_hours),
    'Power input': np.random.uniform(300, 600, n_hours),
    'Tap Weight': np.random.uniform(100, 300, n_hours),
}

df = pd.DataFrame(data)

df.loc[10, 'Sulfur Content'] = 0.12    # way above normal max of 0.05
df.loc[18, 'Furnace_Temp'] = 1480      # too low, below normal min of 1550

df['Timestamp'] = pd.date_range(start='2024-01-01', periods=n_hours, freq='h')

df.to_excel('steel_readings.xlsx', index=False)
print("Excel file saved!")
#print(df.head())



Excel file saved!
(24, 11)


In [9]:
import pandas as pd
import numpy as np

df_input = pd.read_excel('steel_readings.xlsx')
print(df_input.shape)

#to read anomalies, i can add set ranges and check each row within the range by creating a lower bound array and an 
#upper bound array and check if every element is within the range

def detect_anomalies(df):
    anomalies = []

    data_columns = [col for col in df_input.columns if col != 'Timestamp']
    lower_values = np.array([1550.0, 0.02, 0.5, 0.1, 0.3, 0, 0, 20, 300, 100])
    upper_values = np.array([1650.0, 0.8, 3, 0.4, 1.5, 0.05, 0.04, 80, 600, 300])

    for column, lower_value, upper_value in zip(data_columns, lower_values, upper_values):
        low_val = df[df[column] < lower_value]
        upper_val = df[df[column] > upper_value]

        if len(low_val) > 0:
            low_val['issue'] = f'{column} value too low'
            anomalies.append(low_val)
        
        if len(upper_val) > 0:
            upper_val['issue'] = f'{column} value too high'
            anomalies.append(upper_val)

    return pd.concat(anomalies) if anomalies else pd.DataFrame()
            


results = detect_anomalies(df_input)
print(results)

(24, 11)
    Furnace_Temp  Carbon_Content  Oxygen Levels  Silicon Content  \
18   1480.000000        0.387217       0.713841         0.127315   
10   1637.046795        0.105389       1.189269         0.233722   

    Manganese Content  Sulfur Content  Phosphorus Content  Scrap Ratio  \
18           0.448291        0.036243            0.038441    56.985062   
10           0.947918        0.120000            0.012812    75.372251   

    Power input  Tap Weight           Timestamp                          issue  
18   587.118455  292.015128 2024-01-01 18:00:00     Furnace_Temp value too low  
10   554.189596  285.952566 2024-01-01 10:00:00  Sulfur Content value too high  


In [19]:
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import os
import anthropic

load_dotenv()

apiKey = os.getenv("ANTHROPIC_API_KEY")

df_input = pd.read_excel("steel_readings.xlsx");

def detect_anomalies(df):
    anomalies = []

    columns = [col for col in df_input.columns if col != "Timestamp"]
    lower_values = np.array([1550.0, 0.02, 0.5, 0.1, 0.3, 0, 0, 20, 300, 100])
    upper_values = np.array([1650.0, 0.8, 3, 0.4, 1.5, 0.05, 0.04, 80, 600, 300])

    for column, lower_value, upper_value in zip(columns, lower_values, upper_values):
        low_val = df[df[column] < lower_value]
        high_val = df[df[column] > upper_value]

        if len(low_val) > 0:
            low_val = low_val.copy()
            low_val["issue"] = f"{column} has a lower than expected value when it should be between {lower_value} and {upper_value} at {low_val['Timestamp'].values[0]}"
            anomalies.append(low_val)
        if len(high_val) > 0:
            high_val = high_val.copy()
            high_val["issue"] = f"{column} has a higher than expected value when it should be between {lower_value} and {upper_value} at {high_val['Timestamp'].values[0]}"
            anomalies.append(high_val)

    return pd.concat(anomalies) if anomalies else pd.DataFrame()

results = detect_anomalies(df_input)


def get_recommendations(input):
    client = anthropic.Anthropic()

    message = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1024,
    system="You are a blast furnace control room operator in the steel manufacturing business.",
    messages=[
        {"role": "user", 
         "content": f"Based on the data provided here {input}, what would be your recommendation for the blast furnace control room operator. Answer in a user friendly manner where your recommendation will be read by the plant manager"}
    ]
    )
    print(message.content[0].text)


get_recommendations(results.to_string())
    

# BLAST FURNACE OPERATIONAL ALERT & RECOMMENDATIONS

**TO: Plant Manager**
**FROM: Control Room Analysis**
**DATE: 2024-01-01**

---

## **SUMMARY OF ISSUES IDENTIFIED**

We have two operational concerns that require immediate attention:

### **Issue #1: LOW FURNACE TEMPERATURE (18:00 Hours)**
- **Current Temperature:** 1480°C
- **Target Range:** 1550–1650°C
- **Status:** ⚠️ **70°C BELOW MINIMUM**

### **Issue #2: ELEVATED SULFUR CONTENT (10:00 Hours)**
- **Current Level:** 0.120% 
- **Maximum Acceptable:** 0.05%
- **Status:** ⚠️ **2.4x ABOVE SPECIFICATION**

---

## **IMMEDIATE RECOMMENDATIONS**

### **For the Low Temperature Issue:**
1. **Increase power input gradually** (currently at 587 MW) – target 610-630 MW
2. **Improve burden distribution** to enhance heat transfer efficiency
3. **Check fuel quality** – verify coke reactivity and consumption rates
4. **Inspect blast temperature and volume** settings
5. **Monitor in 30-minute intervals** until temperature returns to 1550°C minim